In [ ]:
%reload_ext autoreload
%autoreload 2

# Testing

In [ ]:
# "s2orc_title_abstract": {
#             "args": {"path": "sentence-transformers/s2orc", "split": "train", "name": "title-abstract-pair"},
#             "map_fn": lambda ex: {"anchor": ex["title"], "positive": ex['abstract']},
#             "loss": MultipleNegativesRankingLoss,
#         },




In [ ]:
# from datasets import load_dataset

dataset = load_dataset(
    **{"path": "json", "data_files": "../data_prep/raw/arxiv-metadata-oai-snapshot.json"},
                       num_proc=128
                       )


# "arxiv_title_abstract": {
#             "args": {"path": "json", "data_files"; "../data_prep/raw/arxiv-metadata-oai-snapshot.json"},
#             "map_fn": lambda ex: {"anchor": ex["title"], "positive": ex['abstract']},
#             "loss": MultipleNegativesRankingLoss,
#         },

In [ ]:
import itertools
from datasets import load_dataset, Dataset

# 1. Load the dataset in streaming mode
streaming_dataset = load_dataset(
    **{"path": "../data_prep/raw/pubmed.py", "split": "train", "trust_remote_code": True},
    streaming=True
)

streaming_dataset

In [ ]:
import itertools
from datasets import load_dataset, Dataset

# 1. Load the dataset in streaming mode
streaming_dataset = load_dataset(
    **{"path": "../data_prep/raw/pubmed.py", "split": "train", "trust_remote_code": True},
    streaming=True
)

# 2. Define a function to extract the title and abstract from a record
def extract_text(record):
    """
    Safely extracts 'ArticleTitle' and 'AbstractText' from a nested record.
    Uses .get() to avoid errors if keys are missing.
    """
    # Navigate to the 'Article' dictionary, providing a default empty dict
    article = record.get("MedlineCitation", {}).get("Article", {})

    # Extract the title
    title = article.get("ArticleTitle")

    # Navigate to the 'Abstract' dictionary and then get the text
    abstract_dict = article.get("Abstract", {})
    if isinstance(abstract_dict, dict):
        abstract = abstract_dict.get("AbstractText")
    else:
        # Handles edge cases where 'Abstract' might not be a dictionary
        abstract = None

    return {"title": title, "abstract": abstract}


# 3. Apply the function to your dataset using .map()
# This creates a new generator that will yield the cleaned-up records.
processed_dataset = streaming_dataset.map(extract_text)


# 4. You can now iterate over the processed_dataset.
# Here, we'll take the first 4 elements to show the result.
print("Showing the first 4 processed records:")
for example in itertools.islice(processed_dataset, 4):
    print(example)

In [ ]:
import itertools
from datasets import load_dataset, Dataset

# -- For demonstration purposes, we create a dummy dataset generator --
import itertools
from datasets import load_dataset, Dataset

# 1. Load the dataset in streaming mode
streaming_dataset = load_dataset(
    **{"path": "../data_prep/raw/pubmed.py", "split": "train", "trust_remote_code": True},
    streaming=True
)



# 2. Your existing function to extract the title and abstract from a record
def extract_text(record):
    """
    Safely extracts 'ArticleTitle' and 'AbstractText' from a nested record.
    Uses .get() to avoid errors if keys are missing.
    """
    article = record.get("MedlineCitation", {}).get("Article", {})
    title = article.get("ArticleTitle")
    abstract_dict = article.get("Abstract", {})
    if isinstance(abstract_dict, dict):
        abstract = abstract_dict.get("AbstractText")
    else:
        abstract = None
    return {}


# 3. Your existing map operation to create the processed stream
processed_stream = streaming_dataset.map(extract_text)


# --- NEW: Convert the first 100 records to an in-memory Dataset ---

# 4. Take the first 100 records from the stream and store them in a list
print("Taking the first 100 records from the stream...")
num_records_to_take = 100
records_list = list(itertools.islice(processed_stream, num_records_to_take))
print(f"Collected {len(records_list)} records.")


# 5. Create a new in-memory Dataset from the list of records
in_memory_dataset = Dataset.from_list(records_list)


# 6. Print the new dataset to show its properties (it is no longer streaming)
print("\n--- Created In-Memory Dataset ---")
print(in_memory_dataset)


# 7. You can now use features of a non-streaming dataset, like indexing and len()
print("\n--- Accessing the first record by index ---")
print(in_memory_dataset[0])

print("\n--- Accessing the last record by index ---")
print(in_memory_dataset[99])

In [ ]:
in_memory_dataset

In [ ]:
in_memory_dataset.map()

In [ ]:
from datasets import load_dataset, Dataset
from datasets import (
    Features,
    IterableDataset,
    Sequence,
    Value,
    concatenate_datasets,
    get_dataset_config_names,
    interleave_datasets,
    load_dataset,
    load_from_disk,
)


# Define the features for the new dataset columns
col_union = Features({
    "anchor": Value("string"),
    "positive": Value("string"),
    "negative": Value("string"),
})

raw_ds = load_dataset(
                    **{"path": "../data_prep/raw/pubmed.py", "split": "train"},
                    streaming=False,
                    trust_remote_code=True,
                    features=None,
    )


def process_pubmed_batch(batch):
    """
    Safely processes a batch of PubMed data, handling inconsistent structures
    and filtering out incomplete data points.
    """
    # ====================================================================
    # 1. EXTRACTION - Same as before
    # First, extract all potential data points from the raw batch.
    # ====================================================================
    initial_anchors = []
    initial_positives = []
    
    for item in batch["MedlineCitation"]:
        citation_dict = None
        
        # Universal handler for list or dict inconsistency
        if isinstance(item, list):
            if item:
                citation_dict = item[0]
        elif isinstance(item, dict):
            citation_dict = item

        if not citation_dict:
            initial_anchors.append("")
            initial_positives.append("")
            continue

        # Safely extract Title and Abstract
        title = citation_dict.get("Article", {}).get("ArticleTitle", "")
        initial_anchors.append(title or "")
        
        abstract_data = citation_dict.get("Article", {}).get("Abstract", {}).get("AbstractText")
        if isinstance(abstract_data, list):
            initial_positives.append(" ".join(abstract_data))
        else:
            initial_positives.append(abstract_data or "")

    # ====================================================================
    # 2. FILTERING - The new logic
    # Now, create the final lists, keeping only pairs where BOTH 
    # anchor and positive have content.
    # ====================================================================
    final_anchors = []
    final_positives = []

    for anchor, positive in zip(initial_anchors, initial_positives):
        # The condition: if anchor is not empty AND positive is not empty
        if anchor and positive:
            final_anchors.append(anchor)
            final_positives.append(positive)

    # The 'negative' list should correspond to the final, filtered data.
    final_negatives = [""] * len(final_anchors)
            
    # ====================================================================
    # 3. RETURN - Return the clean, filtered batch
    # ====================================================================
    return {
        "anchor": final_anchors,
        "positive": final_positives,
        "negative": final_negatives,
    }

map_fn = process_pubmed_batch

mapped_ds = raw_ds.map(
                map_fn,
                features=col_union,
                remove_columns=raw_ds.column_names,
                batched=True,  # Usually good for map performance
                batch_size=10,  # Adjust as needed
            )



# Get the top 100 examples as a new IterableDataset
top_100_iterable_ds = mapped_ds.take(100)

# Convert the IterableDataset to a list of dictionaries
top_100_list = list(top_100_iterable_ds)

# Create a Dataset from the list of dictionaries
top_100_ds = Dataset.from_list(top_100_list)

# Now you have a Dataset object with the top 100 examples
# print(top_100_ds)


top_100_ds.to_pandas()

In [ ]:
top_100_ds.to_pandas().iloc[96]["anchor"]

In [ ]:
top_100_ds.to_pandas().describe()

In [ ]:
top_100_ds.to_pandas()["positive"].value_counts()

In [1]:
import pandas as pd

df = pd.read_parquet("/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_st_train_corpus_data.parquet")
df.head(10)

,query,context,type,synthesized,source,metadata,url1
1165247,PaGerE helix-turn-helix structure,Description: In cold and harsh environments su...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2244301233-AMD_KOPRI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
954760,hourly global cloud properties CERES,Description: CER_GEO_Ed4_MET08_NH_V01 is the S...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1237207608-LARC_ASDC"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
894906,chemical and meteorological data East China Sea,Description: NODC Accession 0081044 includes c...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2089376253-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
6085,What was unique about the lunar eclipse mentio...,The shortest total lunar eclipse of the centur...,question-answer,True,SDE_general_v2,"{""id"": ""/SDE/astronomy_picture_of_the_day/|htt...",https://apod.nasa.gov/apod/ap150409.html
1030610,Terra satellite morning passes over equator,Description: MODIS (or Moderate-Resolution Ima...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C3384237152-OB_CLOUD"", ""...",https://cmr.earthdata.nasa.gov/search/concepts...
357642,Geological history of Jezero Crater,NASA Announces Landing Site for Mars 2020 Rove...,search_term-document,True,SDE_general_v3,"{""id"": ""/SDE/astrobiology_at_nasa/|https://ast...",https://astrobiology.nasa.gov/news/nasa-announ...
1020405,ocean temperature assessment methods,Description: Temperature profile and water dep...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2089386562-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
759133,Northern Africa climate reconstruction,Description: This archived Paleoclimatology St...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2103587483-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
1142136,methods for analyzing oceans and water tempera...,Description: Not provided\nPurpose: This datas...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2089381898-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
1206657,Mars Express extended mission 2010-2012 data,Description: This is a Mars Express Radio Scie...,search_term-document,True,PDS,"{""id"": ""/SDE/PDS_API_Legacy_All/|9992bea4740c0...",


In [ ]:
df["source"].value_counts()

In [ ]:
df[["type", "synthesized"]].value_counts()

In [1]:
from datasets import (
    Dataset,
    DatasetDict,
    concatenate_datasets,
    get_dataset_config_names,
    load_dataset,
    load_from_disk,
)

cache_path = "/rhome/sawale/indus_traning/sentense_transformers/data/stage2_cache/NROWS_None/pubmed"

splits = load_from_disk(cache_path)

Loading dataset from disk:   0%|          | 0/39 [00:00<?, ?it/s]

In [2]:
splits

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 24067628
    })
    validation: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1337090
    })
    test: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 1337091
    })
})

In [3]:
dx = splits["train"].to_pandas()

In [5]:
dx

,anchor,positive,negative
0,Air-stable all-inorganic nanocrystal solar cel...,We introduce an ultrathin donor-acceptor solar...,
1,Caring for families with a family history of c...,Care of the family is integral to palliative c...,
2,Mechanisms of biochar assisted immobilization ...,Bioapatite (BAp) is regarded as an effective m...,
3,Nipple discharge: an early warning sign of bre...,Nipple discharge (ND) can be the earliest pres...,
4,Risk Factors for Hospitalization Among Patient...,Among patients with COVID-19 evaluated in outp...,
...,...,...,...
24067623,The miR(21/10b) ratio as a prognostic marker i...,Clear cell renal cell carcinoma (ccRCC) is the...,
24067624,Limb salvage surgery with vascular reconstruct...,Vascular reconstruction and limb salvage surge...,
24067625,Alterations in Child Feeding Behavior: An Unde...,Food allergy (FA) affects around 5.6 million c...,
24067626,Electron and X-ray diffraction studies of infl...,Complexes of influenza virus neuraminidase bot...,


In [ ]:
dx["anchor"].astype(str).unique()

In [ ]:
from sentence_transformers.losses import MultipleNegativesRankingLoss


def build_dataset_configs_s2(N_DATA_SRC=None) -> dict:
    """
    Define all your dataset mappings and losses for stage 2 data.
    """

    def process_pubmed_batch(batch):
        """
        Safely processes a batch of PubMed data, handling inconsistent structures
        and filtering out incomplete data points.
        """
        # ====================================================================
        # 1. EXTRACTION - Same as before
        # First, extract all potential data points from the raw batch.
        # ====================================================================
        initial_anchors = []
        initial_positives = []

        for item in batch["MedlineCitation"]:
            citation_dict = None

            # Universal handler for list or dict inconsistency
            if isinstance(item, list):
                if item:
                    citation_dict = item[0]
            elif isinstance(item, dict):
                citation_dict = item

            if not citation_dict:
                initial_anchors.append("")
                initial_positives.append("")
                continue

            # Safely extract Title and Abstract
            title = citation_dict.get("Article", {}).get("ArticleTitle", "")
            initial_anchors.append(title or "")

            abstract_data = (
                citation_dict.get("Article", {}).get("Abstract", {}).get("AbstractText")
            )
            if isinstance(abstract_data, list):
                initial_positives.append(" ".join(abstract_data))
            else:
                initial_positives.append(abstract_data or "")

        # ====================================================================
        # 2. FILTERING - The new logic
        # Now, create the final lists, keeping only pairs where BOTH
        # anchor and positive have content.
        # ====================================================================
        final_anchors = []
        final_positives = []

        for anchor, positive in zip(initial_anchors, initial_positives):
            # The condition: if anchor is not empty AND positive is not empty
            if anchor and positive:
                final_anchors.append(anchor)
                final_positives.append(positive)

        # The 'negative' list should correspond to the final, filtered data.
        final_negatives = [""] * len(final_anchors)

        # ====================================================================
        # 3. RETURN - Return the clean, filtered batch
        # ====================================================================
        return {
            "anchor": final_anchors,
            "positive": final_positives,
            "negative": final_negatives,
        }

    base = {
        # "nasa-sde-st": {
        #     "args": {"path": "nasa-impact/nasa-sde-st-corpus"},
        #     "map_fn": lambda ex: {"anchor": ex["query"], "positive": ex["context"]},
        #     "loss": MultipleNegativesRankingLoss,
        # },
        "pubmed": {
            "args": {"path": "../data_prep/raw/pubmed.py", "split": "train"},
            "map_fn": process_pubmed_batch,
            "loss": MultipleNegativesRankingLoss,
        }
    }

    if N_DATA_SRC is not None:
        n_src = min(len(base), N_DATA_SRC)
        base = {k: v for i, (k, v) in enumerate(base.items()) if i < n_src}

    return base


In [ ]:
cfg = build_dataset_configs_s2(N_DATA_SRC=100)

In [ ]:
cfg

In [1]:

def process_pubmed_batch(batch):
    """
    Safely processes a batch of PubMed data, handling inconsistent structures
    and filtering out incomplete data points.
    """
    # ====================================================================
    # 1. EXTRACTION - Same as before
    # First, extract all potential data points from the raw batch.
    # ====================================================================
    initial_anchors = []
    initial_positives = []

    for item in batch["MedlineCitation"]:
        citation_dict = None

        # Universal handler for list or dict inconsistency
        if isinstance(item, list):
            if item:
                citation_dict = item[0]
        elif isinstance(item, dict):
            citation_dict = item

        if not citation_dict:
            initial_anchors.append("")
            initial_positives.append("")
            continue

        # Safely extract Title and Abstract
        title = citation_dict.get("Article", {}).get("ArticleTitle", "")
        initial_anchors.append(title or "")

        abstract_data = (
            citation_dict.get("Article", {}).get("Abstract", {}).get("AbstractText")
        )
        if isinstance(abstract_data, list):
            initial_positives.append(" ".join(abstract_data))
        else:
            initial_positives.append(abstract_data or "")

    # ====================================================================
    # 2. FILTERING - The new logic
    # Now, create the final lists, keeping only pairs where BOTH
    # anchor and positive have content.
    # ====================================================================
    final_anchors = []
    final_positives = []

    for anchor, positive in zip(initial_anchors, initial_positives):
        # The condition: if anchor is not empty AND positive is not empty
        if anchor and positive:
            final_anchors.append(anchor)
            final_positives.append(positive)

    # The 'negative' list should correspond to the final, filtered data.
    final_negatives = [""] * len(final_anchors)

    # ====================================================================
    # 3. RETURN - Return the clean, filtered batch
    # ====================================================================
    return {
        "anchor": final_anchors,
        "positive": final_positives,
        "negative": final_negatives,
    }


In [ ]:
from datasets import load_dataset, Dataset
from datasets import (
    Features,
    IterableDataset,
    Sequence,
    Value,
    concatenate_datasets,
    get_dataset_config_names,
    interleave_datasets,
    load_dataset,
    load_from_disk,
)


# Define the features for the new dataset columns
col_union = Features({
    "anchor": Value("string"),
    "positive": Value("string"),
    "negative": Value("string"),
})

raw_ds = load_dataset(
                    **{"path": "../data_prep/raw/pubmed.py", "split": "train"},
                    streaming=False,
                    trust_remote_code=True,
                    features=None,
    )



map_fn = process_pubmed_batch

mapped_ds = raw_ds.map(
                map_fn,
                features=col_union,
                remove_columns=raw_ds.column_names,
                batched=True,  # Usually good for map performance
                batch_size=10,  # Adjust as needed
            )



# Get the top 100 examples as a new IterableDataset
top_100_iterable_ds = mapped_ds.take(100)

# Convert the IterableDataset to a list of dictionaries
top_100_list = list(top_100_iterable_ds)

# Create a Dataset from the list of dictionaries
top_100_ds = Dataset.from_list(top_100_list)

# Now you have a Dataset object with the top 100 examples
# print(top_100_ds)


top_100_ds.to_pandas()

Generating train split: 0 examples [00:00, ? examples/s]

In [17]:
from datasets import load_dataset, Features, Value

# This is the correct map function for your requirement
def process_pubmed(example):
    try:
        title = example['MedlineCitation']['Article']['ArticleTitle']
        abstract = example['MedlineCitation']['Article']['Abstract']['AbstractText']
    except (KeyError, TypeError):
        title, abstract = "", ""
    return {
        "anchor": title,
        "positive": abstract,
        "negative": ""
    }

# --- Your Execution Code ---

# Define the features for the new dataset columns
col_union = Features({
    "anchor": Value("string"),
    "positive": Value("string"),
    "negative": Value("string"),
})

# Load the raw dataset. Shuffling is no longer necessary as we are not sampling negatives.
print("Loading raw dataset...")
raw_ds = load_dataset(
    "../data_prep/raw/pubmed.py", # Assuming your script is here
    split='train',
    trust_remote_code=True,
)

# Apply the updated map function
print("Applying the map function...")
mapped_ds = raw_ds.map(
    process_pubmed, # Use the new function
    # features=col_union,
    remove_columns=raw_ds.column_names,
    num_proc=10,
    # batched=True,
    # batch_size=1000,
)

mapped_ds = mapped_ds.filter(
    lambda example: example["anchor"] and example["positive"]
)

print("Dataset created successfully!")

# Let's check the first record to confirm
print("\nFirst record in the new dataset:")
print(mapped_ds[0])

Loading raw dataset...
Applying the map function...


Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Dataset created successfully!

First record in the new dataset:
{'anchor': "[Biochemical studies on camomile components/III. In vitro studies about the antipeptic activity of (--)-alpha-bisabolol (author's transl)].", 'positive': '(--)-alpha-Bisabolol has a primary antipeptic action depending on dosage, which is not caused by an alteration of the pH-value. The proteolytic activity of pepsin is reduced by 50 percent through addition of bisabolol in the ratio of 1/0.5. The antipeptic action of bisabolol only occurs in case of direct contact. In case of a previous contact with the substrate, the inhibiting effect is lost.', 'negative': ''}


In [18]:
mapped_ds

Dataset({
    features: ['anchor', 'positive', 'negative'],
    num_rows: 56349
})

In [19]:
dx = mapped_ds.to_pandas()

In [20]:
dx

,anchor,positive,negative
0,[Biochemical studies on camomile components/II...,(--)-alpha-Bisabolol has a primary antipeptic ...,
1,[Demonstration of tumor inhibiting properties ...,A report is given on the recent discovery of o...,
2,Influence of a new virostatic compound on the ...,"The virostatic compound N,N-diethyl-4-[2-(2-ox...",
3,Effect of etafenone on total and regional myoc...,The distribution of blood flow to the subendoc...,
4,Pharmacological properties of new neuroleptic ...,"RMI 61 140, RMI 61 144 and RMI 61 280 are newl...",
...,...,...,...
56344,[Hypothalamic-hypophyseal-gonadal axis in chro...,Eleven male chronic alcoholics without cirrhos...,
56345,[Results of radiotherapy of gastrointestinal m...,Irradiation offers only a palliative treatment...,
56346,[Aplastic anemia following indomethacin therapy].,A 78 year old woman developed non-fatal pure r...,
56347,[Physiopathology of prostaglandin I2 synthesis...,1. PGI2 is synthetized in human gastrointestin...,
